## Predictor Variable
<p align="justify">
Now that we have our water quality dataset, the next step is to gather the predictor variables from the <b>Landsat</b> and <b>TerraClimate</b> datasets. In this notebook, we demonstrate how to <b>load previously extracted satellite and climate data</b> from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. Participants can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.
</p>
<p align="justify">
For more detailed guidance on the original data extraction process, you can review the <a href="https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook">Landsat example notebook</a> and the <a href="https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook">TerraClimate example notebook</a> available on the Planetary Computer portal.
</p>

<p align="justify">We have used selected spectral bands — SWIR22 (Shortwave Infrared 2), NIR (Near Infrared), Green, and SWIR16 (Shortwave Infrared 1) — and computed key spectral indices such as NDMI (Normalized Difference Moisture Index) and MNDWI (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, and water content characteristics that influence water quality variability. </p> <p align="justify"> In addition to Landsat features, we also incorporated the <b>Potential Evapotranspiration (PET)</b> variable from the <b>TerraClimate</b> dataset, which provides high-resolution global climate data. The PET feature captures the atmospheric demand for moisture, representing climatic conditions such as temperature, humidity, and radiation that influence surface water evaporation and thus affect water quality parameters. </p> <ul> <li>SWIR22 – Sensitive to surface moisture and turbidity variations in water bodies.</li> <li>NIR – Helps in identifying vegetation and suspended matter in water.</li> <li>Green – Useful for detecting water color and surface reflectance changes.</li> <li>SWIR16 – Provides information on surface dryness and sediment concentration.</li> <li>NDMI – Derived from NIR and SWIR16, indicates moisture and vegetation-water interaction.</li> <li>MNDWI – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.</li> <li>PET – Extracted from the TerraClimate dataset, represents the potential evapotranspiration that influences hydrological and water quality dynamics.</li> </ul>

In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [28]:
import pandas as pd
Water_Quality_df = pd.read_csv('./data/original/water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [29]:
landsat_train_features = pd.read_csv('./data/original/landsat_features_training.csv')
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [30]:
Terraclimate_df = pd.read_csv('./data/original/terraclimate_features_training.csv')
Terraclimate_df.head()

,Latitude,Longitude,Sample Date,pet
0,-28.760833,17.730278,02-01-2011,174.2
1,-26.861111,28.884722,03-01-2011,124.1
2,-26.450000,28.085833,03-01-2011,127.5
3,-27.671111,27.236944,03-01-2011,129.7
4,-27.356667,27.286389,03-01-2011,129.2


In [31]:
from pipeline import *
MERGE_KEYS = ['Latitude', 'Longitude', 'Sample Date']
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df, keys=MERGE_KEYS)

In [32]:
flow_data = pd.read_csv("data/processed/flow_accumulation_locations.csv")
flow_data['flow_accumulation'] = np.log(flow_data['flow_accumulation'])
MERGE_KEYS_1 = ['Latitude', 'Longitude']
wq_data = combine_two_datasets(wq_data, flow_data, keys=MERGE_KEYS_1)

In [33]:
land_data = pd.read_csv("data/processed/land_use_worldcover_1km_locations.csv")
import numpy as np
from sklearn.preprocessing import PowerTransformer
import pandas as pd 

pt = PowerTransformer(method='box-cox', standardize=True)
urban = pd.DataFrame(land_data['pct_urban']) + 1
agri = pd.DataFrame(land_data['pct_agricultural']) + 1

land_data['pct_agricultural'] = pt.fit_transform(agri)
land_data['pct_urban'] = pt.fit_transform(urban)

wq_data = combine_two_datasets(wq_data, land_data, keys=MERGE_KEYS_1)

## Preprocess data

In [34]:
wq_data = wq_data[['swir22','NDMI','MNDWI','pet', "flow_accumulation","pct_agricultural", "pct_urban",  'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

In [35]:
wq_data.head(5)

,swir22,NDMI,MNDWI,pet,flow_accumulation,pct_agricultural,pct_urban,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,7645.0,0.185538,0.195595,174.2,15.234137,-1.548415,-1.235160,128.912,555.0,10.0
1,10574.0,0.124566,-0.180134,124.1,9.360311,1.392921,-0.126133,74.720,162.9,163.0
2,14201.0,-0.083293,-0.252805,127.5,9.103423,1.237099,0.587022,89.254,573.0,80.0
3,11403.0,0.048048,-0.105416,129.7,9.839909,-0.274144,1.822964,82.000,203.6,101.0
4,9643.0,0.141147,-0.142683,129.2,8.471777,1.067431,0.324293,56.100,145.1,151.0


## Run pipeline


In [36]:
X = wq_data.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

y_TA_log = np.log1p(y_TA)
y_EC_log = np.log1p(y_EC)
y_DRP_log = np.log1p(y_DRP)

model_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity", pipeline_kind="simple")
model_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance", pipeline_kind="simple")
model_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus", pipeline_kind="simple")

model_TA_full, results_TA_full = run_pipeline(X, y_TA, "Total Alkalinity", pipeline_kind="full")
model_EC_full, results_EC_full = run_pipeline(X, y_EC, "Electrical Conductance", pipeline_kind="full")
model_DRP_full, results_DRP_full = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus", pipeline_kind="full")


Training Model for Total Alkalinity [pipeline=simple]

Train Evaluation:
R²: 0.954
RMSE: 15.961

Test Evaluation:
R²: 0.814
RMSE: 32.515

Training Model for Electrical Conductance [pipeline=simple]

Train Evaluation:
R²: 0.962
RMSE: 66.907

Test Evaluation:
R²: 0.836
RMSE: 138.167

Training Model for Dissolved Reactive Phosphorus [pipeline=simple]

Train Evaluation:
R²: 0.907
RMSE: 15.545

Test Evaluation:
R²: 0.667
RMSE: 29.575

Training Model for Total Alkalinity [pipeline=full]

Train Evaluation:
R²: 0.785
RMSE: 34.452

Test Evaluation:
R²: 0.767
RMSE: 36.432

Training Model for Electrical Conductance [pipeline=full]

Train Evaluation:
R²: 0.805
RMSE: 151.172

Test Evaluation:
R²: 0.789
RMSE: 156.849

Training Model for Dissolved Reactive Phosphorus [pipeline=full]

Train Evaluation:
R²: 0.563
RMSE: 33.621

Test Evaluation:
R²: 0.543
RMSE: 34.654


## Out-of-fold validation (OOF)

Like the reference code: train one model per fold, fill OOF predictions for the training set, and (optionally) average fold models for test predictions. This gives a more reliable estimate of generalization and can improve scores.

- **`pipeline_kind='simple'`** – benchmark-style: imputer + scaler + RandomForest (no PCA). Often yields higher test R² than the full PCA+XGBoost pipeline.
- **`pipeline_kind='full'`** – current pipeline: imputer + scaler + PCA + XGBoost.
- Pass **`X_test`** to get averaged test predictions from all fold models.

In [37]:
from pipeline import run_pipeline_oof

N_SPLITS = 5
SEED = 42

# OOF with simple (benchmark-style) pipeline – often better test R² than full pipeline
models_TA, oof_TA, results_TA_oof, _ = run_pipeline_oof(X, y_TA_log, n_splits=N_SPLITS, param_name="Total Alkalinity", pipeline_kind='simple', random_state=SEED)
models_EC, oof_EC, results_EC_oof, _ = run_pipeline_oof(X, y_EC_log, n_splits=N_SPLITS, param_name="Electrical Conductance", pipeline_kind='simple', random_state=SEED)
models_DRP, oof_DRP, results_DRP_oof, _ = run_pipeline_oof(X, y_DRP_log, n_splits=N_SPLITS, param_name="Dissolved Reactive Phosphorus", pipeline_kind='simple', random_state=SEED)

# Optional: with held-out X_test, pass it to get averaged test predictions
# models_TA, oof_TA, results_TA_oof, pred_test_TA = run_pipeline_oof(X, y_TA, X_test=X_test, n_splits=N_SPLITS, ...)


OOF Training (5-Fold) for Total Alkalinity [pipeline=simple]
  Fold 1 | R²: 0.8407 | RMSE: 0.346
  Fold 2 | R²: 0.8539 | RMSE: 0.324
  Fold 3 | R²: 0.8589 | RMSE: 0.313
  Fold 4 | R²: 0.8489 | RMSE: 0.327
  Fold 5 | R²: 0.8700 | RMSE: 0.308

  OOF R²: 0.8544 | OOF RMSE: 0.324

OOF Training (5-Fold) for Electrical Conductance [pipeline=simple]
  Fold 1 | R²: 0.8780 | RMSE: 0.287
  Fold 2 | R²: 0.8717 | RMSE: 0.285
  Fold 3 | R²: 0.8787 | RMSE: 0.276
  Fold 4 | R²: 0.8814 | RMSE: 0.278
  Fold 5 | R²: 0.8819 | RMSE: 0.282

  OOF R²: 0.8784 | OOF RMSE: 0.282

OOF Training (5-Fold) for Dissolved Reactive Phosphorus [pipeline=simple]
  Fold 1 | R²: 0.6442 | RMSE: 0.560
  Fold 2 | R²: 0.6165 | RMSE: 0.575
  Fold 3 | R²: 0.6366 | RMSE: 0.566
  Fold 4 | R²: 0.6160 | RMSE: 0.561
  Fold 5 | R²: 0.6035 | RMSE: 0.582

  OOF R²: 0.6238 | OOF RMSE: 0.569
